# LangGraph Agentic Pipeline — Smart Job Search Using Deep Learning

This notebook builds an agentic AI pipeline using LangGraph that connects the fine-tuned Sentence Transformer from Part 2 to live job search APIs, creating an end-to-end resume-to-job matching system.

**Pipeline (4 nodes):**
- **Node 1:** Parse resume using Gemini 3.5 Flash-Lite → extracts name, skills, experience, and job category
- **Node 2:** Generate 3–5 search queries using Gemini based on the parsed profile
- **Node 3:** Fetch live job postings from JSearch API and Adzuna API
- **Node 4:** Score and rank jobs using the fine-tuned all-MiniLM-L6-v2 model by cosine similarity, returning the top 10 matches

**Key details:**
- Built with LangGraph StateGraph — each node reads from and writes to a shared state dictionary
- Pipeline completes in 15–30 seconds
- Uses the fine-tuned model hosted on Hugging Face Hub for scoring
- This pipeline is later integrated into the Streamlit web app (Part 4)

In [ ]:
# Install & Import Libraries

import os
import re
import json
import requests
import numpy as np
from typing import TypedDict, List
from datetime import datetime
 
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
 
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv
import os

load_dotenv()
 
import warnings
warnings.filterwarnings('ignore')
 
print("All libraries imported successfully.")

All libraries imported successfully.


## API Keys and Configuration
Loads API keys from environment variables (.env file) for:
- **Gemini** — powers resume parsing (Node 1) and query generation (Node 2)
- **JSearch (RapidAPI)** — live job search API
- **Adzuna** — secondary job search API for broader coverage

Also loads the fine-tuned Sentence Transformer model for scoring in Node 4.

In [5]:
# API Keys & Configuration

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
JSEARCH_API_KEY = os.getenv("JSEARCH_API_KEY")
ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID")
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")
 
# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-3.5-flash-lite")
 
# Load fine-tuned Sentence Transformer
MODEL_DIR = "fine_tuned_minilm"
st_model = SentenceTransformer(MODEL_DIR)
 
print("Gemini configured.")
print(f"Fine-tuned model loaded from: {MODEL_DIR}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Gemini configured.
Fine-tuned model loaded from: fine_tuned_minilm


## Agent State Definition
Defines the `AgentState` TypedDict that flows through the entire pipeline. Each node reads what it needs from the state, processes it, and writes its output back. Fields include the raw resume text, parsed resume, search queries, fetched jobs, scored results, and any errors encountered.

In [ ]:
# Define Agent State

class AgentState(TypedDict):
    """State that flows through the LangGraph pipeline."""
    resume_text: str
    parsed_resume: dict
    search_queries: List[str]
    jobs: List[dict]
    scored_jobs: List[dict]
    top_matches: List[dict]
    status: str
    errors: List[str]

## Node 1: Parse Resume with Gemini
Sends the resume text to Gemini 3.5 Flash-Lite with a structured prompt asking for JSON output containing:
- Candidate name, email, phone
- Skills, job titles, education
- Estimated years of experience
- Best matching category from the 24 predefined categories
- A 2 3 sentence professional summary

If parsing fails, a fallback dictionary is used so the pipeline can continue.

In [ ]:
# Node 1: Parse Resume with Gemini

def parse_resume(state: AgentState) -> AgentState:
    """Use Gemini to extract structured information from the resume."""
 
    print("\n[Node 1] Parsing resume with Gemini...")
 
    resume_text = state["resume_text"]
 
    prompt = f"""Analyze this resume and extract the following information. 
Respond ONLY in valid JSON format with no markdown backticks or extra text.
 
{{
    "name": "candidate's full name",
    "email": "email if found, otherwise null",
    "phone": "phone if found, otherwise null",
    "skills": ["list", "of", "key", "skills"],
    "experience_years": estimated total years of experience as a number,
    "job_titles": ["list", "of", "previous", "job", "titles"],
    "education": ["list", "of", "degrees", "or", "certifications"],
    "category": "best matching category from this list: ACCOUNTANT, ADVOCATE, AGRICULTURE, APPAREL, ARTS, AUTOMOBILE, AVIATION, BANKING, BPO, BUSINESS-DEVELOPMENT, CHEF, CONSTRUCTION, CONSULTANT, DESIGNER, DIGITAL-MEDIA, ENGINEERING, FINANCE, FITNESS, HEALTHCARE, HR, INFORMATION-TECHNOLOGY, PUBLIC-RELATIONS, SALES, TEACHER",
    "summary": "2-3 sentence professional summary of the candidate"
}}
 
Resume:
{resume_text[:3000]}
"""
 
    try:
        response = gemini_model.generate_content(prompt)
        response_text = response.text.strip()
        response_text = response_text.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(response_text)
 
        print(f"  Name: {parsed.get('name', 'N/A')}")
        print(f"  Category: {parsed.get('category', 'N/A')}")
        print(f"  Skills: {', '.join(parsed.get('skills', [])[:5])}...")
        print(f"  Experience: ~{parsed.get('experience_years', 'N/A')} years")
 
        state["parsed_resume"] = parsed
        state["status"] = "resume_parsed"
 
    except Exception as e:
        print(f"  Error parsing resume: {e}")
        state["errors"].append(f"Resume parsing error: {e}")
        state["parsed_resume"] = {
            "skills": [], "category": "INFORMATION-TECHNOLOGY",
            "summary": resume_text[:200], "name": "Unknown"
        }
        state["status"] = "resume_parse_failed"
 
    return state

## Node 2: Generate Search Queries
Takes the parsed resume profile and asks Gemini to generate 3–5 targeted job search queries (3–6 words each). These queries are tailored to the candidate's specific skills, job titles, and category rather than using generic searches, which improves the relevance of fetched jobs.

In [8]:
#  Node 2: Generate Search Queries
 
def generate_search_queries(state: AgentState) -> AgentState:
    """Use Gemini to generate optimal job search queries based on parsed resume."""
 
    print("\n[Node 2] Generating search queries...")
 
    parsed = state["parsed_resume"]
 
    prompt = f"""Based on this candidate profile, generate 3-5 job search queries 
that would find the best matching job postings. Each query should be 3-6 words.
Respond ONLY as a JSON array of strings, no markdown.
 
Candidate Profile:
- Skills: {', '.join(parsed.get('skills', [])[:10])}
- Job Titles: {', '.join(parsed.get('job_titles', [])[:5])}
- Category: {parsed.get('category', '')}
- Experience: {parsed.get('experience_years', 'N/A')} years
- Summary: {parsed.get('summary', '')}
 
Example output: ["software engineer python", "backend developer", "full stack developer"]
"""
 
    try:
        response = gemini_model.generate_content(prompt)
        response_text = response.text.strip().replace("```json", "").replace("```", "").strip()
        queries = json.loads(response_text)
 
        if not queries:
            queries = [parsed.get("category", "software developer")]
 
        print(f"  Generated {len(queries)} search queries:")
        for q in queries:
            print(f"    - {q}")
 
        state["search_queries"] = queries
        state["status"] = "queries_generated"
 
    except Exception as e:
        print(f"  Error generating queries: {e}")
        state["errors"].append(f"Query generation error: {e}")
        fallback = parsed.get("job_titles", [parsed.get("category", "developer")])
        state["search_queries"] = fallback[:3]
        state["status"] = "queries_fallback"
 
    return state

## Node 3: Search Jobs (JSearch + Adzuna)
Runs each generated query against two job search APIs for broader coverage:
- **JSearch (RapidAPI)** — search-v2 endpoint, filtered to postings from the last month
- **Adzuna** — secondary source with salary data

Results are deduplicated by title and company name. Each job record includes title, company, location, description, salary range, apply link, and source.

In [ ]:
# Node 3: Search Jobs (JSearch + Adzuna)

def search_jsearch(query, api_key, num_results=10):
    """Fetch jobs from JSearch API (RapidAPI)."""
    url = "https://jsearch.p.rapidapi.com/search-v2"
    headers = {
        "x-rapidapi-key": api_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    params = {
        "query": query,
        "page": "1",
        "num_pages": "1",
        "date_posted": "month"
    }

    try:
        response = requests.get(url, headers=headers, params=params, timeout=15)
        data = response.json()
        jobs = []
        for job in data.get("data", {}).get("jobs", [])[:num_results]:
            jobs.append({
                "title": job.get("job_title", "N/A"),
                "company": job.get("employer_name", "N/A"),
                "location": job.get("job_city", "Remote"),
                "description": job.get("job_description", "")[:500],
                "full_description": job.get("job_description", ""),
                "salary_min": job.get("job_min_salary"),
                "salary_max": job.get("job_max_salary"),
                "apply_link": job.get("job_apply_link", ""),
                "source": "JSearch",
                "posted_date": job.get("job_posted_at_datetime_utc", ""),
            })
        return jobs
    except Exception as e:
        print(f"    JSearch error for '{query}': {e}")
        return []
 
 
def search_jobs(state: AgentState) -> AgentState:
    """Search for jobs using multiple APIs based on generated queries."""
 
    print("\n[Node 3] Searching for jobs...")
 
    queries = state["search_queries"]
    all_jobs = []
    seen_titles = set()
 
    for query in queries:
        print(f"  Searching: '{query}'")
 
        jsearch_jobs = search_jsearch(query, JSEARCH_API_KEY, num_results=5)
        print(f"    JSearch: {len(jsearch_jobs)} results")
 
        adzuna_jobs = search_adzuna(query, ADZUNA_APP_ID, ADZUNA_APP_KEY, num_results=5)
        print(f"    Adzuna: {len(adzuna_jobs)} results")
 
        for job in jsearch_jobs + adzuna_jobs:
            key = f"{job['title'].lower().strip()}_{job['company'].lower().strip()}"
            if key not in seen_titles:
                seen_titles.add(key)
                job["search_query"] = query
                all_jobs.append(job)
 
    print(f"\n  Total unique jobs found: {len(all_jobs)}")
 
    state["jobs"] = all_jobs
    state["status"] = "jobs_found" if all_jobs else "no_jobs_found"
 
    return state

## Node 4: Score Matches with Fine-Tuned Model
The core matching step that uses the fine-tuned Sentence Transformer from Part 2:
- Combines the candidate's skills, experience, and summary into a single resume embedding
- Encodes each job description into an embedding
- Computes cosine similarity between the resume and every job
- Sorts by similarity score and returns the top 10 matches

This is where the deep learning model trained in Parts 1 and 2 is applied to real job data.

In [ ]:
# Node 4: Score Matches with Fine-Tuned Model

def clean_text_for_model(text):
    """Light cleaning matching the fine-tuning preprocessing."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(text.split()[:256])
 
 
def score_matches(state: AgentState) -> AgentState:
    """Score each job against the resume using the fine-tuned Sentence Transformer."""
 
    print("\n[Node 4] Scoring job matches with fine-tuned model...")
 
    resume_text = state["resume_text"]
    jobs = state["jobs"]
 
    if not jobs:
        print("  No jobs to score.")
        state["scored_jobs"] = []
        state["status"] = "no_jobs_to_score"
        return state
 
    clean_resume = clean_text_for_model(resume_text)
    resume_embedding = st_model.encode([clean_resume])
 
    job_texts = [
        clean_text_for_model(job.get("full_description", job.get("description", "")))
        for job in jobs
    ]
    job_embeddings = st_model.encode(job_texts, show_progress_bar=False)
 
    similarities = cos_sim(resume_embedding, job_embeddings)[0]
 
    scored_jobs = []
    for i, job in enumerate(jobs):
        job_copy = job.copy()
        job_copy["similarity_score"] = round(float(similarities[i]), 4)
        scored_jobs.append(job_copy)
 
    scored_jobs.sort(key=lambda x: x["similarity_score"], reverse=True)
    top_10 = scored_jobs[:10]
 
    print(f"  Scored {len(jobs)} jobs. Top 10:")
    for i, job in enumerate(top_10, 1):
        print(f"    {i:2d}. [{job['similarity_score']:.3f}] {job['title']} at {job['company']}")
 
    state["scored_jobs"] = top_10
    state["status"] = "jobs_scored"
 
    return state

## Build the LangGraph Pipeline
Assembles the four nodes into a LangGraph StateGraph with a linear flow:

`Parse Resume → Generate Queries → Search Jobs → Score Matches → END`

The compiled pipeline is a callable function — pass in an initial state with resume text and it returns the final state with scored jobs.

In [19]:
# Build the LangGraph Pipeline

def build_pipeline():
    """Construct the LangGraph state machine."""

    workflow = StateGraph(AgentState)

    workflow.add_node("parse_resume", parse_resume)
    workflow.add_node("generate_queries", generate_search_queries)
    workflow.add_node("search_jobs", search_jobs)
    workflow.add_node("score_matches", score_matches)

    workflow.set_entry_point("parse_resume")
    workflow.add_edge("parse_resume", "generate_queries")
    workflow.add_edge("generate_queries", "search_jobs")
    workflow.add_edge("search_jobs", "score_matches")
    workflow.add_edge("score_matches", END)

    pipeline = workflow.compile()

    print("LangGraph pipeline built successfully!")
    print("\n  Pipeline flow:")
    print("  Parse Resume --> Generate Queries --> Search Jobs --> Score Matches")

    return pipeline


pipeline = build_pipeline()

LangGraph pipeline built successfully!

  Pipeline flow:
  Parse Resume --> Generate Queries --> Search Jobs --> Score Matches


In [12]:
# Display Results

def display_results(result):
    """Pretty print the pipeline results."""

    parsed = result.get("parsed_resume", {})
    scored_jobs = result.get("scored_jobs", [])

    print("\n" + "=" * 70)
    print("          SMART JOB SEARCH — RESULTS")
    print("=" * 70)

    print(f"\n  Candidate: {parsed.get('name', 'N/A')}")
    print(f"  Category:  {parsed.get('category', 'N/A')}")
    print(f"  Experience: {parsed.get('experience_years', 'N/A')} years")
    print(f"  Skills: {', '.join(parsed.get('skills', [])[:8])}")

    print(f"\n  {'='*66}")
    print(f"  TOP {len(scored_jobs)} JOB MATCHES")
    print(f"  {'='*66}")

    for i, job in enumerate(scored_jobs, 1):
        score = job.get("similarity_score", 0)
        if job.get("salary_min") and job.get("salary_max"):
            salary = f"${job['salary_min']:,.0f} - ${job['salary_max']:,.0f}"
        elif job.get("salary_min"):
            salary = f"${job['salary_min']:,.0f}+"
        else:
            salary = "Not listed"

        print(f"\n  {'-'*66}")
        print(f"  #{i} | Match Score: {score:.3f} | Source: {job.get('source', 'N/A')}")
        print(f"  {'-'*66}")
        print(f"  Title:    {job['title']}")
        print(f"  Company:  {job['company']}")
        print(f"  Location: {job.get('location', 'N/A')}")
        print(f"  Salary:   {salary}")
        print(f"  Apply:    {job.get('apply_link', 'N/A')[:80]}")

    print(f"\n  {'='*66}")
    print(f"  Pipeline Status: {result.get('status', 'N/A')}")
    if result.get("errors"):
        print(f"  Errors: {len(result['errors'])}")
        for err in result["errors"]:
            print(f"    - {err}")
    print(f"  {'='*66}")

## Run the Pipeline
Tests the full pipeline end-to-end with a sample software engineer resume. The pipeline runs through all four nodes in sequence, printing progress at each step, and displays the top 10 matched jobs with similarity scores, salary information, and apply links. Typical runtime is 15–30 seconds.

In [20]:
# Run the Pipeline (Test with Sample Resume)

 
sample_resume = """
John Smith
john.smith@email.com | (555) 123-4567 | San Diego, CA
 
PROFESSIONAL SUMMARY
Experienced software engineer with 5 years of experience in full-stack development.
Proficient in Python, JavaScript, React, and cloud technologies. Strong background
in building scalable web applications and RESTful APIs.
 
EXPERIENCE
Senior Software Engineer | TechCorp Inc. | 2022 - Present
- Developed and maintained microservices using Python and FastAPI
- Built front-end applications with React and TypeScript
- Deployed applications on AWS using Docker and Kubernetes
- Implemented CI/CD pipelines with GitHub Actions
 
Software Developer | WebSolutions LLC | 2020 - 2022
- Built full-stack web applications using Django and React
- Designed and optimized PostgreSQL databases
- Collaborated with cross-functional teams using Agile methodology
 
Junior Developer | StartupXYZ | 2019 - 2020
- Developed RESTful APIs using Flask
- Created automated testing suites with pytest
- Participated in code reviews and sprint planning
 
EDUCATION
Master of Science in Computer Science | University of San Diego | 2019
Bachelor of Science in Information Technology | State University | 2017
 
SKILLS
Python, JavaScript, TypeScript, React, Django, Flask, FastAPI, PostgreSQL,
MongoDB, AWS, Docker, Kubernetes, Git, CI/CD, Agile, REST APIs, GraphQL
"""
 
print("Starting Smart Job Search Pipeline...")
print("=" * 70)

initial_state = {
    "resume_text": sample_resume,
    "parsed_resume": {},
    "search_queries": [],
    "jobs": [],
    "scored_jobs": [],
    "status": "started",
    "errors": [],
}

result = pipeline.invoke(initial_state)
display_results(result)

Starting Smart Job Search Pipeline...

[Node 1] Parsing resume with Gemini...
  Name: John Smith
  Category: INFORMATION-TECHNOLOGY
  Skills: Python, JavaScript, TypeScript, React, Django...
  Experience: ~5 years

[Node 2] Generating search queries...
  Generated 4 search queries:
    - senior python software engineer
    - full stack developer react python
    - senior software developer aws
    - python react web developer

[Node 3] Searching for jobs...
  Searching: 'senior python software engineer'
    JSearch: 5 results
    Adzuna: 5 results
  Searching: 'full stack developer react python'
    JSearch: 5 results
    Adzuna: 5 results
  Searching: 'senior software developer aws'
    JSearch: 5 results
    Adzuna: 5 results
  Searching: 'python react web developer'
    JSearch: 5 results
    Adzuna: 5 results

  Total unique jobs found: 31

[Node 4] Scoring job matches with fine-tuned model...
  Scored 31 jobs. Top 10:
     1. [0.615] Full Stack Software Developer (Python/FastAPI/R